# 🌏 NZ Earthquake Declustering — Deep Learning Autoencoder
## Unsupervised Crisis vs Non-Crisis Classification from Scratch

---

### 💡 What This Notebook Does

We train an **Autoencoder** neural network on your 22 earthquake features.

The key idea is simple:

> **Background (non-crisis) events follow a regular pattern — the network learns this pattern well.**  
> **Crisis events (aftershocks, swarms) are unusual — the network struggles to reconstruct them → HIGH ERROR**

So reconstruction error becomes our **anomaly score**.

---

### 🏗️ Architecture (Simple Explanation)

```
INPUT (22 features)
    ↓
ENCODER — compresses 22 → 8 → 4  (learns "what is normal")
    ↓
BOTTLENECK — 4 numbers capture the whole event
    ↓
DECODER — reconstructs 4 → 8 → 22  (tries to rebuild original)
    ↓
COMPARE INPUT vs RECONSTRUCTION → Reconstruction Error
    ↓
High Error  → CRISIS   (unusual event, network can't reconstruct it)
Low  Error  → NON-CRISIS (normal background, network knows this pattern)
```

---

### 📋 Your 22 Features Used

| Feature | Meaning |
|---|---|
| T1 – T10 | Time distances to 10 nearest-neighbour events |
| R1 – R10 | Space distances to 10 nearest-neighbour events |
| Mn | Magnitude ratio (STA/LTA style — how 'large' relative to neighbours) |
| bval | Local b-value over neighbourhood window |

---

### 📦 Required Libraries
```
pip install torch scikit-learn pandas numpy matplotlib seaborn
```


## Step 1 — Import Libraries

In [ ]:
# ── Standard libraries ────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Scikit-learn (preprocessing + evaluation) ─────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report

# ── PyTorch (neural network) ──────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ── Setup ─────────────────────────────────────────────────────
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'✅ PyTorch version : {torch.__version__}')
print(f'✅ Device          : {DEVICE}')
print(f'✅ Output folder   : {OUTPUT_DIR.resolve()}')


## Step 2 — Configuration (All Tunable Settings in One Place)

> ✏️ **Change only this cell to tune the entire pipeline.**


In [ ]:
# ════════════════════════════════════════════════════════════
#   CONFIGURATION  —  edit these values to tune the model
# ════════════════════════════════════════════════════════════

# ── File path ────────────────────────────────────────────────
CSV_PATH = 'som_feature_nz_real_catalog.csv'   # ← change to your file

# ── Feature columns (22 features from the KDM paper) ─────────
T_COLS       = [f'T{i}' for i in range(1, 11)]   # T1 … T10
R_COLS       = [f'R{i}' for i in range(1, 11)]   # R1 … R10
EXTRA_COLS   = ['Mn', 'bval']
FEATURE_COLS = T_COLS + R_COLS + EXTRA_COLS       # 22 features total

# ── Autoencoder architecture ─────────────────────────────────
LATENT_DIM   = 4      # bottleneck size  (try: 2, 4, 8)
HIDDEN_DIMS  = [64, 32, 16]   # encoder layer sizes (decoder mirrors these)

# ── Training ─────────────────────────────────────────────────
BATCH_SIZE    = 2048   # events per batch
N_EPOCHS      = 60     # training epochs  (increase to 100 for better results)
LEARNING_RATE = 1e-3   # Adam optimiser learning rate
WEIGHT_DECAY  = 1e-5   # L2 regularisation

# ── Classification threshold ─────────────────────────────────
# Events with reconstruction error ABOVE this percentile = crisis
CRISIS_PERCENTILE = 75   # top 25% errors → crisis  (try: 70, 75, 80)

# ── Reproducibility ──────────────────────────────────────────
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('✅ Configuration set')
print(f'   Features    : {len(FEATURE_COLS)} ({FEATURE_COLS})')
print(f'   Architecture: {len(FEATURE_COLS)} → {HIDDEN_DIMS} → {LATENT_DIM} (bottleneck)')
print(f'   Batch size  : {BATCH_SIZE}')
print(f'   Epochs      : {N_EPOCHS}')
print(f'   Threshold   : top {100 - CRISIS_PERCENTILE}% reconstruction error = crisis')


## Step 3 — Load & Inspect Data

We load the catalogue and verify all 22 features are present.


In [ ]:
# ── Load CSV ──────────────────────────────────────────────────
print(f'Loading: {CSV_PATH}')
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f'Raw dataset shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')


## Step 4 — Clean Data

Remove rows with missing features or invalid neighbour info (`i+` = 0 means no neighbours found).


In [ ]:
# ── Check all 22 features exist ───────────────────────────────
missing_cols = [c for c in FEATURE_COLS if c not in df_raw.columns]
if missing_cols:
    print(f'❌ Missing columns: {missing_cols}')
    raise ValueError('Fix column names in FEATURE_COLS config above')
else:
    print(f'✅ All {len(FEATURE_COLS)} feature columns found')

# ── Drop rows with any NaN in the 22 features ─────────────────
df = df_raw.dropna(subset=FEATURE_COLS).copy()
print(f'After dropping NaN rows: {len(df):,} events (removed {len(df_raw)-len(df):,})')

# ── Drop events with no spatial neighbours (i+ = 0) ──────────
if 'i+' in df.columns:
    before = len(df)
    df = df[df['i+'] > 0].copy()
    print(f'After removing no-neighbour events (i+=0): {len(df):,} (removed {before-len(df):,})')

# ── Reset index ───────────────────────────────────────────────
df = df.reset_index(drop=True)
print(f'\n✅ Clean dataset: {len(df):,} events ready for training')

# Quick stats on the 22 features
print('\nFeature statistics (first 5 rows):')
df[FEATURE_COLS].describe().round(4)


## Step 5 — Build Feature Matrix & Normalise

Neural networks need all inputs on the same scale.  
We use **StandardScaler** → each feature becomes mean=0, std=1.

> Why StandardScaler (not MinMax)?  
> Because your T and R columns span very different ranges and  
> StandardScaler handles outliers better for density-based data.


In [ ]:
# ── Extract 22 features as numpy array ───────────────────────
X_raw = df[FEATURE_COLS].values.astype(np.float32)

# Replace any inf/-inf with 0 (safety step)
X_raw = np.nan_to_num(X_raw, nan=0.0, posinf=0.0, neginf=0.0)

print(f'Feature matrix shape: {X_raw.shape}')   # should be (n_events, 22)
print(f'Memory: {X_raw.nbytes / 1e6:.1f} MB')

# ── Normalise: StandardScaler ─────────────────────────────────
#   fit_transform() computes mean & std from data, then transforms it
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw).astype(np.float32)

# Verify: mean ≈ 0, std ≈ 1 for each feature
print(f'\nAfter scaling:')
print(f'  Mean range : {X_scaled.mean(axis=0).min():.4f} to {X_scaled.mean(axis=0).max():.4f}  (should be ~0)')
print(f'  Std  range : {X_scaled.std(axis=0).min():.4f} to {X_scaled.std(axis=0).max():.4f}   (should be ~1)')

# ── Plot feature distributions before and after scaling ───────
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Before
axes[0].boxplot(X_raw, labels=FEATURE_COLS, vert=False, patch_artist=True,
                boxprops=dict(facecolor='lightcoral', alpha=0.7))
axes[0].set_title('Feature Distributions — Before Scaling (raw)', fontsize=12)
axes[0].set_xlabel('Value')
axes[0].tick_params(axis='y', labelsize=8)

# After
axes[1].boxplot(X_scaled, labels=FEATURE_COLS, vert=False, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[1].set_title('Feature Distributions — After StandardScaler', fontsize=12)
axes[1].set_xlabel('Standardised Value')
axes[1].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'AE_01_feature_distributions.png'), dpi=150)
plt.show()
print('✅ Feature matrix ready')


## Step 6 — Create PyTorch DataLoader

We wrap the feature matrix in a PyTorch `DataLoader`.  
This handles **batching + shuffling** automatically during training.


In [ ]:
# ── Convert numpy array → PyTorch tensor ─────────────────────
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

# ── TensorDataset wraps the tensor so DataLoader can use it ──
dataset = TensorDataset(X_tensor)

# ── DataLoader: batches + shuffles during training ────────────
loader = DataLoader(
    dataset,
    batch_size = BATCH_SIZE,
    shuffle    = True,    # shuffle every epoch for better training
    num_workers= 0,       # 0 = use main process (safe on all OS)
    drop_last  = False    # keep last (smaller) batch
)

n_batches = len(loader)
print(f'✅ DataLoader ready')
print(f'   Total events   : {len(dataset):,}')
print(f'   Batch size     : {BATCH_SIZE}')
print(f'   Batches/epoch  : {n_batches}')
print(f'   Total features : {X_tensor.shape[1]}')


## Step 7 — Define the Autoencoder

### Architecture Explained

```
INPUT (22)  ──►  Encoder  ──►  Bottleneck (4)  ──►  Decoder  ──►  OUTPUT (22)
               [64→32→16]                         [16→32→64]
```

**Each layer has:**
- `Linear` — learns weights (the actual computation)
- `BatchNorm1d` — stabilises training by normalising activations
- `ReLU` — non-linear activation (allows learning complex patterns)

**The bottleneck (4 neurons)** is the key:  
The network is *forced* to compress 22 features into just 4 numbers.  
It learns to keep only the most important information.


In [ ]:
class SeismicAutoencoder(nn.Module):
    """
    Autoencoder for earthquake feature learning.

    The encoder compresses 22 features down to LATENT_DIM.
    The decoder reconstructs back to 22.
    Reconstruction error = anomaly score.
    """

    def __init__(self, input_dim, hidden_dims, latent_dim):
        super(SeismicAutoencoder, self).__init__()

        # ── BUILD ENCODER ─────────────────────────────────────
        # Goes: input_dim → hidden_dims[0] → hidden_dims[1] → ... → latent_dim
        encoder_layers = []
        prev_dim = input_dim

        for h_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, h_dim),   # fully connected layer
                nn.BatchNorm1d(h_dim),         # normalise activations
                nn.ReLU()                      # non-linear activation
            ])
            prev_dim = h_dim

        # Final encoder layer: compress to latent space (NO activation here)
        encoder_layers.append(nn.Linear(prev_dim, latent_dim))
        self.encoder = nn.Sequential(*encoder_layers)

        # ── BUILD DECODER ─────────────────────────────────────
        # Mirrors the encoder: latent_dim → ... → input_dim
        decoder_layers = []
        prev_dim = latent_dim

        for h_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU()
            ])
            prev_dim = h_dim

        # Final decoder layer: reconstruct to original input size
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        """
        Forward pass: input → encoder → bottleneck → decoder → output
        Returns: (reconstructed_x, latent_code)
        """
        latent        = self.encoder(x)    # compress
        reconstructed = self.decoder(latent)  # reconstruct
        return reconstructed, latent

    def encode(self, x):
        """Get just the latent (compressed) representation."""
        return self.encoder(x)


# ── Instantiate the model ─────────────────────────────────────
INPUT_DIM = X_scaled.shape[1]   # = 22

model = SeismicAutoencoder(
    input_dim   = INPUT_DIM,
    hidden_dims = HIDDEN_DIMS,
    latent_dim  = LATENT_DIM
).to(DEVICE)

# ── Print model summary ───────────────────────────────────────
print('Model architecture:')
print(model)
print()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')
print(f'Input → Latent: {INPUT_DIM}D → {LATENT_DIM}D')


## Step 8 — Train the Autoencoder

### What happens each epoch:
1. Feed a batch of events through the network
2. Get the reconstruction (what the network thinks the input should be)
3. Compute **MSE loss** = how wrong the reconstruction is
4. Backpropagate the error → update weights to improve

### Loss function: Mean Squared Error (MSE)
```
MSE = average of (predicted_value - true_value)²  over all 22 features
```
We want this to go **as low as possible**.

### Learning rate schedule:
After epoch 20 and 40, we **reduce the learning rate** to fine-tune.  
This is like first taking big steps then small careful steps.

> ⏱️ **Expected time on workstation:** ~5–15 minutes (CPU) | ~2–5 min (GPU)


In [ ]:
# ── Loss function: Mean Squared Error ─────────────────────────
criterion = nn.MSELoss()

# ── Optimiser: Adam (adaptive learning rates) ─────────────────
optimizer = optim.Adam(
    model.parameters(),
    lr           = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY
)

# ── Learning rate scheduler: reduce LR at epochs 20 and 40 ────
scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones = [20, 40],   # reduce LR at these epochs
    gamma      = 0.3         # multiply LR by 0.3
)

# ── Training loop ─────────────────────────────────────────────
train_losses = []
print(f'Starting training on {DEVICE}...')
print(f'{"Epoch":>6} | {"Loss":>12} | {"LR":>10}')
print('-' * 35)

for epoch in range(1, N_EPOCHS + 1):

    model.train()       # put model in training mode
    epoch_loss = 0.0

    for (batch_x,) in loader:
        batch_x = batch_x.to(DEVICE)   # move to GPU if available

        # ── Forward pass ──────────────────────────────────────
        reconstructed, _ = model(batch_x)

        # ── Compute loss ──────────────────────────────────────
        loss = criterion(reconstructed, batch_x)

        # ── Backward pass (compute gradients) ─────────────────
        optimizer.zero_grad()   # clear old gradients
        loss.backward()         # compute new gradients

        # ── Update weights ────────────────────────────────────
        optimizer.step()

        epoch_loss += loss.item()

    # Average loss for this epoch
    avg_loss = epoch_loss / len(loader)
    train_losses.append(avg_loss)
    current_lr = scheduler.get_last_lr()[0]

    # Update learning rate
    scheduler.step()

    # Print progress every 5 epochs
    if epoch % 5 == 0 or epoch == 1:
        print(f'{epoch:>6} | {avg_loss:>12.6f} | {current_lr:>10.2e}')

print()
print(f'✅ Training complete!')
print(f'   Initial loss : {train_losses[0]:.6f}')
print(f'   Final loss   : {train_losses[-1]:.6f}')
print(f'   Improvement  : {100*(train_losses[0]-train_losses[-1])/train_losses[0]:.1f}%')

# ── Save model weights ────────────────────────────────────────
model_path = str(OUTPUT_DIR / 'NZ_autoencoder_model.pt')
torch.save(model.state_dict(), model_path)
print(f'✅ Model saved → {model_path}')


## Step 9 — Visualise Training Curve

A smoothly decreasing loss means the network is learning well.  
If loss plateaus early → increase epochs.  
If loss is noisy → reduce learning rate.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full training curve
axes[0].plot(range(1, N_EPOCHS+1), train_losses, 'b-', lw=1.5)
axes[0].set_title('Training Loss Over Epochs', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].grid(alpha=0.3)
# Mark LR reductions
for milestone in [20, 40]:
    if milestone < N_EPOCHS:
        axes[0].axvline(milestone, color='red', linestyle='--', alpha=0.5,
                        label=f'LR reduced at epoch {milestone}')
axes[0].legend(fontsize=9)

# Log scale (easier to see convergence)
axes[1].plot(range(1, N_EPOCHS+1), train_losses, 'b-', lw=1.5)
axes[1].set_yscale('log')
axes[1].set_title('Training Loss (Log Scale)', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss (log)')
axes[1].grid(alpha=0.3)

plt.suptitle('Autoencoder Training Progress — NZ Catalogue', fontsize=13)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'AE_02_training_curve.png'), dpi=150)
plt.show()


## Step 10 — Compute Reconstruction Error for Every Event

After training, we pass ALL events through the network and measure how well each one is reconstructed.

```
Per-event error = mean( (original_features - reconstructed_features)² )
```

- **Small error** → the network knows this pattern → **non-crisis (background)**
- **Large error** → the network never learned this pattern → **crisis (aftershock/swarm)**


In [ ]:
model.eval()   # put model in evaluation mode (disables BatchNorm training behaviour)

all_errors  = []   # reconstruction error per event
all_latents = []   # latent code per event (the 4 compressed values)

print(f'Computing reconstruction errors for {len(X_scaled):,} events...')

with torch.no_grad():   # no gradient computation needed (saves memory + time)
    for i in range(0, len(X_scaled), BATCH_SIZE):

        # Get current batch (numpy → tensor → device)
        batch_np = X_scaled[i : i + BATCH_SIZE]
        batch    = torch.tensor(batch_np, dtype=torch.float32).to(DEVICE)

        # Forward pass
        reconstructed, latent = model(batch)

        # Per-event MSE: mean over all 22 features
        # Shape: (batch_size, 22) → mean over dim=1 → (batch_size,)
        per_event_error = ((reconstructed - batch) ** 2).mean(dim=1)

        all_errors.append(per_event_error.cpu().numpy())
        all_latents.append(latent.cpu().numpy())

# Stack all batches into single arrays
recon_errors = np.concatenate(all_errors)    # shape: (n_events,)
latent_codes = np.concatenate(all_latents)   # shape: (n_events, LATENT_DIM)

print(f'\n✅ Reconstruction errors computed')
print(f'   Shape        : {recon_errors.shape}')
print(f'   Min error    : {recon_errors.min():.6f}')
print(f'   Mean error   : {recon_errors.mean():.6f}')
print(f'   Max error    : {recon_errors.max():.6f}')
print(f'   75th pctile  : {np.percentile(recon_errors, 75):.6f}')
print(f'   90th pctile  : {np.percentile(recon_errors, 90):.6f}')


## Step 11 — Explore the Error Distribution

A good autoencoder shows a **bimodal distribution** (two humps):
- Left hump = non-crisis events (well-reconstructed)
- Right hump = crisis events (poorly reconstructed)

If you see one hump → the model may need more training or a smaller LATENT_DIM.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Raw error histogram ───────────────────────────────────────
axes[0].hist(recon_errors, bins=100, color='steelblue', alpha=0.8, edgecolor='white')
threshold_val = np.percentile(recon_errors, CRISIS_PERCENTILE)
axes[0].axvline(threshold_val, color='red', linestyle='--', lw=2,
                label=f'Threshold ({CRISIS_PERCENTILE}th pctile) = {threshold_val:.4f}')
axes[0].set_title('Reconstruction Error Distribution', fontsize=12)
axes[0].set_xlabel('MSE per event')
axes[0].set_ylabel('Number of events')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── Log-scale for better visibility ──────────────────────────
axes[1].hist(recon_errors, bins=100, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].axvline(threshold_val, color='red', linestyle='--', lw=2)
axes[1].set_yscale('log')
axes[1].set_title('Reconstruction Error (Log Scale)', fontsize=12)
axes[1].set_xlabel('MSE per event')
axes[1].set_ylabel('Count (log)')
axes[1].grid(alpha=0.3)

# ── Percentile curve ─────────────────────────────────────────
percentiles = np.arange(0, 101, 1)
pctile_vals = np.percentile(recon_errors, percentiles)
axes[2].plot(percentiles, pctile_vals, 'b-', lw=2)
axes[2].axhline(threshold_val, color='red', linestyle='--', lw=2,
                label=f'{CRISIS_PERCENTILE}th percentile threshold')
axes[2].axvline(CRISIS_PERCENTILE, color='red', linestyle=':', alpha=0.7)
axes[2].set_title('Percentile Curve of Errors', fontsize=12)
axes[2].set_xlabel('Percentile')
axes[2].set_ylabel('Reconstruction Error')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('Reconstruction Error Analysis — NZ Catalogue', fontsize=14)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'AE_03_error_distribution.png'), dpi=150)
plt.show()

print(f'Threshold value: {threshold_val:.6f}')
print(f'Events below threshold (non-crisis): {(recon_errors < threshold_val).sum():,}')
print(f'Events above threshold (crisis)    : {(recon_errors >= threshold_val).sum():,}')


## Step 12 — Classify Events: Crisis vs Non-Crisis

**How we convert reconstruction error → probability:**

1. Take log of error (handles the heavy-tailed distribution)
2. Scale to [0, 1] range using min-max
3. This gives us P(crisis) — how 'unusual' each event is

**Hard label:**
- P(crisis) ≥ 0.5 → `crisis`
- P(crisis) < 0.5 → `non_crisis`


In [ ]:
# ── Convert error to P(crisis) ───────────────────────────────
# Step 1: log-transform to reduce skewness
log_errors = np.log1p(recon_errors)   # log(1 + error), safe for 0-values

# Step 2: scale to [0, 1]
e_min = log_errors.min()
e_max = log_errors.max()
P_crisis = (log_errors - e_min) / (e_max - e_min + 1e-10)

P_non_crisis = 1.0 - P_crisis

# ── Hard label: crisis if P_crisis >= 0.5 ────────────────────
labels = np.where(P_crisis >= 0.5, 'crisis', 'non_crisis')

# ── Confidence: how decisive the classification is ───────────
# 0 = completely uncertain (P=0.5), 1 = fully certain (P=0 or 1)
confidence = np.abs(P_crisis - 0.5) / 0.5

# ── Add everything to the dataframe ──────────────────────────
df_result = df.copy()
df_result['recon_error']  = recon_errors
df_result['P_crisis']     = P_crisis
df_result['P_non_crisis'] = P_non_crisis
df_result['label']        = labels
df_result['confidence']   = confidence

# ── Summary ───────────────────────────────────────────────────
n_total   = len(df_result)
n_crisis  = (df_result['label'] == 'crisis').sum()
n_bg      = (df_result['label'] == 'non_crisis').sum()
n_hconf   = (df_result['confidence'] > 0.8).sum()
n_lconf   = (df_result['confidence'] < 0.2).sum()

print('══ Classification Summary ═══════════════════════════')
print(f'  Total events         : {n_total:,}')
print(f'  Crisis events        : {n_crisis:,}  ({100*n_crisis/n_total:.1f}%)')
print(f'  Non-crisis events    : {n_bg:,}  ({100*n_bg/n_total:.1f}%)')
print(f'  High confidence (>0.8): {n_hconf:,}  ({100*n_hconf/n_total:.1f}%)')
print(f'  Uncertain (<0.2 conf) : {n_lconf:,}  ({100*n_lconf/n_total:.1f}%)')
print('═════════════════════════════════════════════════════')
print()
df_result[['latitude','longitude','magnitude','recon_error',
           'P_crisis','label','confidence']].head(10)


## Step 13 — Explore the Latent Space

The 4D bottleneck captures what the network learned.  
We use PCA to project it to 2D for visualisation.

A good latent space shows **visible separation** between crisis and non-crisis events.


In [ ]:
# ── PCA on latent codes: 4D → 2D for plotting ─────────────────
pca = PCA(n_components=2, random_state=RANDOM_SEED)
latent_2d = pca.fit_transform(latent_codes)

var_explained = pca.explained_variance_ratio_
print(f'PCA variance explained: PC1={var_explained[0]:.1%}  PC2={var_explained[1]:.1%}  '
      f'Total={sum(var_explained):.1%}')

# Subsample for fast plotting if very large dataset
MAX_PLOT = 50_000
if len(latent_2d) > MAX_PLOT:
    idx_plot = np.random.choice(len(latent_2d), MAX_PLOT, replace=False)
    print(f'(Plotting subsample of {MAX_PLOT:,} events for speed)')
else:
    idx_plot = np.arange(len(latent_2d))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ── Panel 1: Crisis vs Non-crisis labels ─────────────────────
crisis_mask = df_result['label'].values[idx_plot] == 'crisis'
nc_mask     = ~crisis_mask

axes[0].scatter(latent_2d[idx_plot][nc_mask, 0],
                latent_2d[idx_plot][nc_mask, 1],
                c='steelblue', s=0.8, alpha=0.4, label=f'Non-crisis')
axes[0].scatter(latent_2d[idx_plot][crisis_mask, 0],
                latent_2d[idx_plot][crisis_mask, 1],
                c='crimson', s=0.8, alpha=0.4, label=f'Crisis')
axes[0].set_title('Latent Space — Crisis vs Non-Crisis', fontsize=12)
axes[0].set_xlabel(f'PC1 ({var_explained[0]:.1%} var)')
axes[0].set_ylabel(f'PC2 ({var_explained[1]:.1%} var)')
axes[0].legend(markerscale=6, fontsize=10)

# ── Panel 2: Reconstruction error ────────────────────────────
sc1 = axes[1].scatter(latent_2d[idx_plot, 0],
                       latent_2d[idx_plot, 1],
                       c=np.log1p(recon_errors[idx_plot]),
                       cmap='YlOrRd', s=0.8, alpha=0.5)
plt.colorbar(sc1, ax=axes[1], label='log(reconstruction error)')
axes[1].set_title('Latent Space — Reconstruction Error', fontsize=12)
axes[1].set_xlabel(f'PC1 ({var_explained[0]:.1%} var)')
axes[1].set_ylabel(f'PC2 ({var_explained[1]:.1%} var)')

# ── Panel 3: P(crisis) ───────────────────────────────────────
sc2 = axes[2].scatter(latent_2d[idx_plot, 0],
                       latent_2d[idx_plot, 1],
                       c=P_crisis[idx_plot],
                       cmap='RdYlGn_r', s=0.8, alpha=0.5,
                       vmin=0, vmax=1)
plt.colorbar(sc2, ax=axes[2], label='P(crisis)')
axes[2].set_title('Latent Space — P(crisis)', fontsize=12)
axes[2].set_xlabel(f'PC1 ({var_explained[0]:.1%} var)')
axes[2].set_ylabel(f'PC2 ({var_explained[1]:.1%} var)')

plt.suptitle('Autoencoder Latent Space Visualisation — NZ Catalogue', fontsize=14)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'AE_04_latent_space.png'), dpi=150)
plt.show()


## Step 14 — Feature-Level Reconstruction Quality

Check which of the 22 features are reconstructed well vs. poorly.  
Features with high error contribute most to the anomaly detection signal.


In [ ]:
model.eval()
all_recon_perfeature = []

with torch.no_grad():
    for i in range(0, len(X_scaled), BATCH_SIZE):
        batch = torch.tensor(X_scaled[i:i+BATCH_SIZE], dtype=torch.float32).to(DEVICE)
        recon, _ = model(batch)
        # Per-feature error: shape (batch_size, 22)
        per_feat = ((recon - batch) ** 2).cpu().numpy()
        all_recon_perfeature.append(per_feat)

per_feature_errors = np.concatenate(all_recon_perfeature, axis=0)  # (n_events, 22)
mean_per_feature   = per_feature_errors.mean(axis=0)               # (22,)

# ── For crisis vs non-crisis separately ──────────────────────
crisis_idx  = df_result['label'].values == 'crisis'
nc_idx      = df_result['label'].values == 'non_crisis'

crisis_feat_err = per_feature_errors[crisis_idx].mean(axis=0)
nc_feat_err     = per_feature_errors[nc_idx].mean(axis=0)

# ── Plot ─────────────────────────────────────────────────────
x_pos = np.arange(len(FEATURE_COLS))
fig, axes = plt.subplots(2, 1, figsize=(16, 9))

# Overall
axes[0].bar(x_pos, mean_per_feature, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(FEATURE_COLS, rotation=45, ha='right', fontsize=9)
axes[0].set_title('Mean Reconstruction Error per Feature (All Events)', fontsize=12)
axes[0].set_ylabel('Mean Squared Error')
axes[0].grid(axis='y', alpha=0.3)

# Crisis vs non-crisis per feature
width = 0.4
axes[1].bar(x_pos - width/2, crisis_feat_err, width,
            label='Crisis',     color='crimson',   alpha=0.8, edgecolor='white')
axes[1].bar(x_pos + width/2, nc_feat_err, width,
            label='Non-crisis', color='steelblue', alpha=0.8, edgecolor='white')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(FEATURE_COLS, rotation=45, ha='right', fontsize=9)
axes[1].set_title('Reconstruction Error per Feature: Crisis vs Non-Crisis', fontsize=12)
axes[1].set_ylabel('Mean Squared Error')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'AE_05_feature_errors.png'), dpi=150)
plt.show()

# Top 5 most discriminative features
diff = crisis_feat_err - nc_feat_err
top5_idx = np.argsort(diff)[::-1][:5]
print('Top 5 features most different between crisis and non-crisis:')
for i, idx in enumerate(top5_idx, 1):
    print(f'  {i}. {FEATURE_COLS[idx]:6s}  '
          f'crisis_err={crisis_feat_err[idx]:.4f}  '
          f'nc_err={nc_feat_err[idx]:.4f}  '
          f'diff={diff[idx]:.4f}')


## Step 15 — Spatial Map of Classification Results

Plot crisis vs non-crisis events on the NZ map.  
Also plot P(crisis) as a continuous colour gradient.


In [ ]:
crisis_df     = df_result[df_result['label'] == 'crisis']
non_crisis_df = df_result[df_result['label'] == 'non_crisis']

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# ── Hard labels ───────────────────────────────────────────────
axes[0].scatter(non_crisis_df['longitude'], non_crisis_df['latitude'],
                c='steelblue', s=0.5, alpha=0.3,
                label=f'Non-crisis: {len(non_crisis_df):,}')
axes[0].scatter(crisis_df['longitude'], crisis_df['latitude'],
                c='crimson', s=0.5, alpha=0.3,
                label=f'Crisis: {len(crisis_df):,}')
axes[0].set_title('Autoencoder — Crisis vs Non-Crisis (NZ)', fontsize=12)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].legend(markerscale=8, fontsize=10)
axes[0].grid(alpha=0.2)

# ── Continuous P(crisis) ─────────────────────────────────────
sc = axes[1].scatter(df_result['longitude'], df_result['latitude'],
                     c=df_result['P_crisis'], cmap='RdYlGn_r',
                     s=0.5, alpha=0.4, vmin=0, vmax=1)
cbar = plt.colorbar(sc, ax=axes[1])
cbar.set_label('P(crisis)', fontsize=11)
axes[1].set_title('P(crisis) Continuous Map — NZ', fontsize=12)
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].grid(alpha=0.2)

plt.suptitle('Autoencoder Spatial Declustering — New Zealand Catalogue', fontsize=14)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'AE_06_spatial_map.png'), dpi=150)
plt.show()


## Step 16 — Cumulative Curves & Magnitude Distribution

**What to look for:**
- **Non-crisis curve** → should be nearly linear (steady background rate)
- **Crisis curve** → should show staircase jumps at major NZ earthquake sequences (Kaikōura 2016, Darfield 2010, etc.)


In [ ]:
time_col = 'time' if 'time' in df_result.columns else 'Year'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Sort by time ──────────────────────────────────────────────
df_sorted     = df_result.sort_values(time_col).reset_index(drop=True)
crisis_sorted = df_sorted[df_sorted['label'] == 'crisis']
nc_sorted     = df_sorted[df_sorted['label'] == 'non_crisis']
N             = len(df_sorted)

# ── Cumulative curves ─────────────────────────────────────────
axes[0].plot(df_sorted[time_col],
             np.arange(N) / N,
             'k--', lw=1, alpha=0.5, label='Full catalogue')
axes[0].plot(crisis_sorted[time_col],
             np.arange(len(crisis_sorted)) / N,
             color='crimson', linestyle=':', lw=2,
             label=f'Crisis ({len(crisis_sorted):,})')
axes[0].plot(nc_sorted[time_col],
             np.arange(len(nc_sorted)) / N,
             color='steelblue', linestyle='-', lw=2,
             label=f'Non-crisis ({len(nc_sorted):,})')
axes[0].set_title('Cumulative Event Curves', fontsize=12)
axes[0].set_xlabel('Time (decimal year)')
axes[0].set_ylabel('Proportion of events')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# ── Magnitude distribution ────────────────────────────────────
bins = np.arange(0, df_result['magnitude'].max() + 0.5, 0.3)
axes[1].hist(crisis_sorted['magnitude'],
             bins=bins, density=True, alpha=0.65,
             color='crimson', label='Crisis', edgecolor='white')
axes[1].hist(nc_sorted['magnitude'],
             bins=bins, density=True, alpha=0.65,
             color='steelblue', label='Non-crisis', edgecolor='white')
axes[1].set_title('Magnitude Distribution', fontsize=12)
axes[1].set_xlabel('Magnitude')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.suptitle('Autoencoder Results — NZ Earthquake Catalogue', fontsize=14)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'AE_07_cumulative_magnitude.png'), dpi=150)
plt.show()

# Stats
print(f'Mean magnitude — Crisis    : {crisis_sorted["magnitude"].mean():.2f}')
print(f'Mean magnitude — Non-crisis: {nc_sorted["magnitude"].mean():.2f}')


## Step 17 — Depth & P(crisis) Analysis

Examine how crisis probability varies with depth and magnitude.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── P(crisis) vs magnitude ────────────────────────────────────
axes[0].hexbin(df_result['magnitude'], df_result['P_crisis'],
               gridsize=40, cmap='YlOrRd', mincnt=1)
axes[0].set_xlabel('Magnitude')
axes[0].set_ylabel('P(crisis)')
axes[0].set_title('P(crisis) vs Magnitude', fontsize=12)

# ── P(crisis) vs depth ───────────────────────────────────────
if 'depth' in df_result.columns:
    depth_clip = df_result['depth'].clip(0, 200)   # clip extreme depths
    axes[1].hexbin(depth_clip, df_result['P_crisis'],
                   gridsize=40, cmap='YlOrRd', mincnt=1)
    axes[1].set_xlabel('Depth (km, clipped at 200)')
    axes[1].set_ylabel('P(crisis)')
    axes[1].set_title('P(crisis) vs Depth', fontsize=12)
else:
    axes[1].text(0.5, 0.5, 'No depth column found', ha='center', va='center',
                 transform=axes[1].transAxes)

# ── Confidence distribution ───────────────────────────────────
axes[2].hist(df_result['confidence'], bins=50, color='purple',
             alpha=0.8, edgecolor='white')
axes[2].axvline(0.8, color='red', linestyle='--', label='High confidence threshold')
axes[2].set_xlabel('Confidence')
axes[2].set_ylabel('Number of events')
axes[2].set_title('Classification Confidence Distribution', fontsize=12)
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'AE_08_depth_magnitude_analysis.png'), dpi=150)
plt.show()


## Step 18 — Threshold Sensitivity Analysis

Test different thresholds to see how the crisis/non-crisis split changes.  
Choose the threshold that makes physical sense for NZ seismicity.

> **Expected for NZ:** ~65–80% crisis events (subduction zone = high aftershock rate)


In [ ]:
thresholds = [60, 65, 70, 75, 80, 85, 90]
results_thresh = []

for pct in thresholds:
    thresh    = np.percentile(recon_errors, pct)
    n_crisis  = (recon_errors >= thresh).sum()
    n_bg      = (recon_errors < thresh).sum()
    results_thresh.append({
        'percentile'  : pct,
        'threshold'   : round(thresh, 6),
        'n_crisis'    : n_crisis,
        'n_non_crisis': n_bg,
        'pct_crisis'  : round(100 * n_crisis / len(recon_errors), 1)
    })

df_thresh = pd.DataFrame(results_thresh)

# ── Print table ───────────────────────────────────────────────
print('Threshold Sensitivity:')
print(df_thresh.to_string(index=False))
print(f'\nCurrent setting: CRISIS_PERCENTILE = {CRISIS_PERCENTILE}')

# ── Plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_thresh['percentile'], df_thresh['pct_crisis'], 'ro-', lw=2)
axes[0].axvline(CRISIS_PERCENTILE, color='blue', linestyle='--',
                label=f'Current: {CRISIS_PERCENTILE}th pctile')
axes[0].set_xlabel('Threshold Percentile')
axes[0].set_ylabel('% Events Labelled Crisis')
axes[0].set_title('% Crisis vs Threshold Percentile', fontsize=12)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(df_thresh['percentile'].astype(str),
            df_thresh['n_crisis'], label='Crisis', color='crimson', alpha=0.8)
axes[1].bar(df_thresh['percentile'].astype(str),
            df_thresh['n_non_crisis'],
            bottom=df_thresh['n_crisis'], label='Non-crisis', color='steelblue', alpha=0.8)
axes[1].set_xlabel('Threshold Percentile')
axes[1].set_ylabel('Number of events')
axes[1].set_title('Event Counts by Threshold', fontsize=12)
axes[1].legend()

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'AE_09_threshold_sensitivity.png'), dpi=150)
plt.show()


## Step 19 — Save Final Results

Output columns in the saved CSV:

| Column | Description |
|---|---|
| `recon_error` | Raw reconstruction MSE for this event |
| `P_crisis` | Probability this is a crisis event (0–1) |
| `P_non_crisis` | Probability this is a background event (0–1) |
| `label` | Hard label: `crisis` or `non_crisis` |
| `confidence` | How decisive the classification is (0=uncertain, 1=certain) |
| `latent_0…3` | The 4 compressed feature values from the bottleneck |


In [ ]:
# ── Add latent dimensions to output ──────────────────────────
for i in range(LATENT_DIM):
    df_result[f'latent_{i}'] = latent_codes[:, i]

# ── Select output columns ─────────────────────────────────────
keep_base = ['event', 'DateTime', 'latitude', 'longitude',
             'depth', 'magnitude', 'time', 'Year']
keep_base = [c for c in keep_base if c in df_result.columns]

keep_new  = ['recon_error', 'P_crisis', 'P_non_crisis', 'label', 'confidence']
keep_new += [f'latent_{i}' for i in range(LATENT_DIM)]

output_cols = keep_base + keep_new

# ── Save ─────────────────────────────────────────────────────
out_path = str(OUTPUT_DIR / 'NZ_Autoencoder_declustered.csv')
df_result[output_cols].to_csv(out_path, index=False)

n_c  = (df_result['label'] == 'crisis').sum()
n_nc = (df_result['label'] == 'non_crisis').sum()

print(f'✅ Saved → {out_path}')
print(f'   Rows    : {len(df_result):,}')
print(f'   Columns : {len(output_cols)}')
print()
print('══ FINAL SUMMARY ═══════════════════════════════════')
print(f'  Total events         : {len(df_result):,}')
print(f'  Crisis               : {n_c:,}  ({100*n_c/len(df_result):.1f}%)')
print(f'  Non-crisis           : {n_nc:,}  ({100*n_nc/len(df_result):.1f}%)')
print(f'  High conf (>0.8)     : {(df_result["confidence"]>0.8).sum():,}  '
      f'({100*(df_result["confidence"]>0.8).sum()/len(df_result):.1f}%)')
print(f'  Model final loss     : {train_losses[-1]:.6f}')
print(f'  Architecture         : {INPUT_DIM} → {HIDDEN_DIMS} → {LATENT_DIM} → {list(reversed(HIDDEN_DIMS))} → {INPUT_DIM}')
print(f'  Threshold percentile : {CRISIS_PERCENTILE}th')
print('═════════════════════════════════════════════════════')


## Step 20 — How to Reload the Model Later (Optional)

If you want to apply the trained model to new events without retraining:


In [ ]:
# ── How to reload saved model ────────────────────────────────
# Run this in a new session to reload without retraining

# 1. Recreate model with same architecture
model_reload = SeismicAutoencoder(
    input_dim   = INPUT_DIM,
    hidden_dims = HIDDEN_DIMS,
    latent_dim  = LATENT_DIM
).to(DEVICE)

# 2. Load saved weights
model_reload.load_state_dict(
    torch.load(str(OUTPUT_DIR / 'NZ_autoencoder_model.pt'),
               map_location=DEVICE)
)
model_reload.eval()
print('✅ Model reloaded from disk')

# 3. Apply to new data
# new_X_scaled = scaler.transform(new_features)   # use SAME scaler!
# new_tensor   = torch.tensor(new_X_scaled, dtype=torch.float32)
# with torch.no_grad():
#     new_recon, new_latent = model_reload(new_tensor)
#     new_errors = ((new_recon - new_tensor)**2).mean(dim=1).numpy()

print()
print('📌 Important: always use the SAME scaler (StandardScaler) fitted on training data')
print('   The scaler parameters are stored in the `scaler` variable')
print('   Save it with: import joblib; joblib.dump(scaler, "outputs/scaler.pkl")')


## Step 21 — Tuning Guide

| Problem | Solution |
|---|---|
| Loss not decreasing | Increase `N_EPOCHS` to 100–150 |
| Loss decreasing but slowly | Increase `LEARNING_RATE` to 3e-3 |
| Loss very noisy | Increase `BATCH_SIZE` to 4096 |
| Too many crisis events (>85%) | Increase `CRISIS_PERCENTILE` to 80 or 85 |
| Too few crisis events (<50%) | Decrease `CRISIS_PERCENTILE` to 65 or 70 |
| Poor latent space separation | Try `LATENT_DIM = 2` (forces more compression) |
| Want richer representation | Try `LATENT_DIM = 8` or `HIDDEN_DIMS = [128, 64, 32]` |
| Out of memory | Reduce `BATCH_SIZE` to 512 or 1024 |
| Training too slow | Set `N_EPOCHS = 30` for quick test |

### 📊 Outputs Generated

| File | Contents |
|---|---|
| `AE_01_feature_distributions.png` | Raw vs scaled features |
| `AE_02_training_curve.png` | Loss convergence |
| `AE_03_error_distribution.png` | Reconstruction error histogram |
| `AE_04_latent_space.png` | 4D latent space in 2D (PCA) |
| `AE_05_feature_errors.png` | Per-feature reconstruction error |
| `AE_06_spatial_map.png` | NZ map with crisis/non-crisis |
| `AE_07_cumulative_magnitude.png` | Cumulative curves + magnitude |
| `AE_08_depth_magnitude_analysis.png` | P(crisis) vs depth/magnitude |
| `AE_09_threshold_sensitivity.png` | Effect of different thresholds |
| `NZ_Autoencoder_declustered.csv` | Final labelled catalogue |
| `NZ_autoencoder_model.pt` | Saved model weights |
